# Baseline MobileNetV2-UNet trên Kaggle — 1 GPU, train và đánh giá đầy đủ

Notebook tự chứa toàn bộ pipeline và đọc trực tiếp dataset `tinlngnguyn/skin-lesion-segmentation-hus` đã Attach tại `/kaggle/input`. Training chính thức được chia 3 fold; LR Finder chỉ dùng train subset, augmentation chỉ dùng cho train subset và checkpoint được chọn bằng Best Dice trên fold holdout sạch.

Sau khi cả 3 fold đã được khóa, notebook đánh giá từng fold model và equal-weight ensemble trên **official validation**, rồi đánh giá single model đã chọn từ CV và ensemble trên **official test**. Validation/test tuyệt đối không tham gia LR Finder, train hoặc chọn checkpoint.

> Trên Kaggle: `Add Input` dataset `tinlngnguyn/skin-lesion-segmentation-hus`, chọn Accelerator GPU (một GPU là đủ) và bật Internet nếu cache MobileNetV2 ImageNet chưa có. Artifact được ghi vào `/kaggle/working/mobilenetv2_unet_kaggle_1gpu`.

In [ ]:
# Kaggle image đã có PyTorch, torchvision, OpenCV và Albumentations.
# Không pip install giữa session để tránh thay đổi binary/package đang chạy.
print('Using the Kaggle runtime environment.')

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from scipy import ndimage
from torch.utils.data import DataLoader, Dataset, get_worker_info
from torchvision import models
from tqdm.auto import tqdm

try:
    import psutil
except ImportError:
    psutil = None

# -------------------- Cấu hình baseline --------------------
DATASET_HANDLE = 'tinlngnguyn/skin-lesion-segmentation-hus'
DATA_ROOT = Path('/kaggle/input/datasets/tinlngnguyn/skin-lesion-segmentation-hus')
SPLIT_DIRECTORIES = {
    'train': ('ISIC2018_Task1-2_Training_Input', 'ISIC2018_Task1_Training_GroundTruth'),
    'validation': ('ISIC2018_Task1-2_Validation_Input', 'ISIC2018_Task1_Validation_GroundTruth'),
    'test': ('ISIC2018_Task1-2_Test_Input', 'ISIC2018_Task1_Test_GroundTruth'),
}
EXPECTED_COUNTS = {'train': 2594, 'validation': 100, 'test': 1000}
SEED = 42
N_FOLDS = 3
SIZES = (128, 256)
FROZEN_EPOCHS = 30
UNFROZEN_EPOCHS = 30
PER_GPU_BATCH_BY_SIZE = {128: 16, 256: 8}
NUM_WORKERS = min(2, os.cpu_count() or 1)
PREFETCH_FACTOR = 1
PIN_MEMORY = True
CACHE_RESIZED_IN_RAM = True
DROP_OS_PAGE_CACHE = True
SAVE_LAST_EVERY = 5
LRF_ITERS = 50
LR_MIN, LR_MAX, LRF_BETA = 1e-6, 1.0, 0.9
LR_SELECTION_FACTOR = 0.1
MAX_LR_FROZEN, MAX_LR_UNFROZEN = 3e-3, 1e-3
CUT_FRAC, STLR_RATIO = 0.1, 32.0
WEIGHT_DECAY = 0.0
GRAD_CLIP_NORM = 1.0
PRED_THRESHOLD = 0.5
JACCARD_CUT = 0.65
PRETRAINED = True
USE_CHANNELS_LAST = False
FP32_TRANSPOSED_CONV = False  # MobileNetV2 decoder dùng bilinear upsample, không dùng ConvTranspose2d
USE_FUSED_ADAM = False
RESUME = True
FAST_DEV_RUN = False
PROTOCOL_VERSION = 3

if FAST_DEV_RUN:
    SIZES = (128,)
    FROZEN_EPOCHS = UNFROZEN_EPOCHS = 1
    LRF_ITERS = 8

if not hasattr(A, 'D4'):
    raise RuntimeError('Albumentations runtime quá cũ: cần A.D4 và A.Compose(seed=...).')
try:
    A.Compose([], seed=SEED)
except TypeError as error:
    raise RuntimeError('Albumentations runtime không hỗ trợ Compose(seed=...).') from error

OUTPUT_ROOT = Path('/kaggle/working/mobilenetv2_unet_kaggle_1gpu')
RUN_NAME = 'fast_dev_v3' if FAST_DEV_RUN else 'full_v3'
RUN_DIR = OUTPUT_ROOT / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / 'checkpoints'
LOG_DIR = RUN_DIR / 'logs'
for directory in (RUN_DIR, CHECKPOINT_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

set_seed()
cv2.setNumThreads(0)
torch.set_float32_matmul_precision('high')
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật Accelerator GPU trong Kaggle Settings.')

DEVICE = torch.device('cuda:0')
GPU_COUNT = torch.cuda.device_count()
GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]
BATCH_SIZE_BY_SIZE = dict(PER_GPU_BATCH_BY_SIZE)  # notebook luôn chỉ dùng cuda:0
USE_AMP = True
print({
    'device': str(DEVICE), 'gpus': GPU_NAMES, 'global_batch': BATCH_SIZE_BY_SIZE,
    'workers': NUM_WORKERS, 'channels_last': USE_CHANNELS_LAST,
    'fp32_upconv': FP32_TRANSPOSED_CONV, 'run_dir': str(RUN_DIR),
})

def memory_status():
    parts = []
    if psutil is not None:
        process = psutil.Process(os.getpid())
        info = process.memory_full_info()
        main_bytes = getattr(info, 'uss', info.rss)
        worker_bytes = 0
        for child in process.children(recursive=True):
            try:
                child_info = child.memory_full_info()
                worker_bytes += getattr(child_info, 'uss', child_info.rss)
            except psutil.Error:
                pass
        parts.extend([f'ram_main={main_bytes / 2**30:.2f}GiB',
                      f'ram_workers={worker_bytes / 2**30:.2f}GiB'])
    allocated = torch.cuda.memory_allocated(0)
    reserved = torch.cuda.memory_reserved(0)
    parts.extend([f'cuda_alloc={allocated / 2**30:.2f}GiB',
                  f'cuda_reserved={reserved / 2**30:.2f}GiB'])
    return ' '.join(parts)

In [ ]:
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png'}

def find_populated_directory(directory_name, suffixes):
    outer = DATA_ROOT / directory_name
    candidates = [outer / directory_name, outer]
    if DATA_ROOT.is_dir():
        candidates.extend(path for path in DATA_ROOT.rglob(directory_name) if path.is_dir())
    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen or not candidate.is_dir():
            continue
        seen.add(candidate)
        if any(path.is_file() and path.suffix.lower() in suffixes
               for path in candidate.iterdir()):
            return candidate
    raise FileNotFoundError(f'Không tìm thấy thư mục dữ liệu {directory_name} dưới {DATA_ROOT}')

def discover_records(split):
    image_name, mask_name = SPLIT_DIRECTORIES[split]
    image_dir = find_populated_directory(image_name, IMAGE_SUFFIXES)
    mask_dir = find_populated_directory(mask_name, {'.png'})
    image_paths = [path for path in image_dir.iterdir()
                   if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES]
    mask_paths = [path for path in mask_dir.iterdir()
                  if path.is_file() and path.suffix.lower() == '.png'
                  and path.stem.endswith('_segmentation')]
    if len({path.stem for path in image_paths}) != len(image_paths):
        raise RuntimeError(f'{split}: trùng image ID')
    image_map = {path.stem: path for path in image_paths}
    mask_map = {path.stem.removesuffix('_segmentation'): path for path in mask_paths}
    if len(mask_map) != len(mask_paths):
        raise RuntimeError(f'{split}: trùng mask ID')
    missing_masks = sorted(set(image_map) - set(mask_map))
    orphan_masks = sorted(set(mask_map) - set(image_map))
    if missing_masks or orphan_masks:
        raise RuntimeError(f'{split}: thiếu mask={missing_masks[:5]}, mask mồ côi={orphan_masks[:5]}')
    records = [
        {'image_id': image_id, 'image_path': image_map[image_id],
         'mask_path': mask_map[image_id]}
        for image_id in sorted(image_map)
    ]
    if len(records) != EXPECTED_COUNTS[split]:
        raise RuntimeError(f'{split}: cần {EXPECTED_COUNTS[split]} cặp, thấy {len(records)}')
    return records, image_dir, mask_dir

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f'Chưa Attach dataset {DATASET_HANDLE}; không thấy {DATA_ROOT}'
    )
train_records_all, train_image_dir, train_mask_dir = discover_records('train')
official_val_records_all, val_image_dir, val_mask_dir = discover_records('validation')
test_records_all, test_image_dir, test_mask_dir = discover_records('test')

split_ids = {
    'train': {record['image_id'] for record in train_records_all},
    'validation': {record['image_id'] for record in official_val_records_all},
    'test': {record['image_id'] for record in test_records_all},
}
assert split_ids['train'].isdisjoint(split_ids['validation'])
assert split_ids['train'].isdisjoint(split_ids['test'])
assert split_ids['validation'].isdisjoint(split_ids['test'])

def deterministic_subset(records, count, seed):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(len(records))[:min(count, len(records))]
    return [records[int(index)] for index in indices]

if FAST_DEV_RUN:
    train_records = deterministic_subset(train_records_all, 96, SEED)
    official_val_records = deterministic_subset(official_val_records_all, 24, SEED + 1)
    test_records = deterministic_subset(test_records_all, 24, SEED + 2)
else:
    train_records = train_records_all
    official_val_records = official_val_records_all
    test_records = test_records_all

generator = torch.Generator().manual_seed(SEED)
permutation = torch.randperm(len(train_records), generator=generator).tolist()
fold_sizes = [len(train_records) // N_FOLDS] * N_FOLDS
for index in range(len(train_records) % N_FOLDS):
    fold_sizes[index] += 1
fold_indices, cursor = [], 0
for fold_size in fold_sizes:
    fold_indices.append(permutation[cursor:cursor + fold_size])
    cursor += fold_size
folds, all_indices = [], set(range(len(train_records)))
for holdout_indices in fold_indices:
    holdout_set = set(holdout_indices)
    train_indices = sorted(all_indices - holdout_set)
    train_subset = [train_records[index] for index in train_indices]
    holdout_subset = [train_records[index] for index in holdout_indices]
    assert {r['image_id'] for r in train_subset}.isdisjoint(
        r['image_id'] for r in holdout_subset
    )
    folds.append((train_subset, holdout_subset))

print('pairs:', {'train': len(train_records), 'validation': len(official_val_records),
                 'test': len(test_records)}, 'fold sizes:', fold_sizes)
print('train dirs:', train_image_dir, train_mask_dir, sep='\n  ')
print('validation dirs:', val_image_dir, val_mask_dir, sep='\n  ')
print('test dirs:', test_image_dir, test_mask_dir, sep='\n  ')

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
RAM_CACHE = {}

def gray_world(image, **kwargs):
    del kwargs
    image_float = image.astype(np.float32)
    channel_means = image_float.mean(axis=(0, 1))
    gray_mean = channel_means.mean()
    scale = gray_mean / np.maximum(channel_means, 1e-6)
    return np.clip(image_float * scale.reshape(1, 1, 3), 0, 255).astype(np.uint8)

def load_pil_uint8(path, mode):
    with open(path, 'rb') as handle:
        with Image.open(handle) as source:
            array = np.asarray(source.convert(mode), dtype=np.uint8).copy()
        if (DROP_OS_PAGE_CACHE and hasattr(os, 'posix_fadvise')
                and hasattr(os, 'POSIX_FADV_DONTNEED')):
            try:
                os.posix_fadvise(handle.fileno(), 0, 0, os.POSIX_FADV_DONTNEED)
            except OSError:
                pass
    return array

def build_transform(size, train, seed, preprocessed=False):
    steps = []
    if not preprocessed:
        steps.extend([
            A.Resize(size, size, interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST),
            A.Lambda(image=gray_world, p=1.0),
        ])
    if train:
        steps.extend([
            A.D4(p=1.0),
            A.Affine(rotate=(-44, 44), scale=(1.0, 1.05),
                     interpolation=cv2.INTER_LINEAR,
                     mask_interpolation=cv2.INTER_NEAREST, p=0.75),
            A.RandomBrightnessContrast(brightness_limit=0.2,
                                       contrast_limit=0.2, p=0.75),
        ])
    steps.append(ToTensorV2())
    return A.Compose(steps, seed=int(seed))

class ISICDataset(Dataset):
    def __init__(self, records, size, train, seed=SEED):
        if not records:
            raise ValueError('Dataset rỗng')
        self.records = list(records)
        self.size = int(size)
        self.train = bool(train)
        cache = RAM_CACHE.get(self.size)
        indices = None if cache is None else [cache['id_to_index'].get(r['image_id'])
                                              for r in self.records]
        self.cache = cache if indices is not None and all(i is not None for i in indices) else None
        self.cache_indices = indices if self.cache is not None else None
        self.transform = build_transform(
            self.size, self.train, seed, preprocessed=self.cache is not None
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        if self.cache is not None:
            cache_index = self.cache_indices[index]
            image_np = self.cache['images'][cache_index].copy()
            mask_np = self.cache['masks'][cache_index].copy()
        else:
            image_np = load_pil_uint8(record['image_path'], 'RGB')
            mask_np = load_pil_uint8(record['mask_path'], 'L')
            if image_np.shape[:2] != mask_np.shape:
                raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
            mask_np = (mask_np > 127).astype(np.uint8)
        transformed = self.transform(image=image_np, mask=mask_np)
        image, mask = transformed['image'], transformed['mask']
        if mask.ndim == 2:
            mask = mask.unsqueeze(0)
        return image.contiguous(), (mask > 0).float().contiguous()

def seed_worker(worker_id):
    del worker_id
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    worker = get_worker_info()
    if worker is not None and hasattr(worker.dataset.transform, 'set_random_seed'):
        worker.dataset.transform.set_random_seed(worker_seed)

def make_loader(records, size, train, seed, batch_size=None, drop_last=False):
    if batch_size is None:
        batch_size = BATCH_SIZE_BY_SIZE[int(size)]
    kwargs = dict(
        dataset=ISICDataset(records, size, train, seed),
        batch_size=int(batch_size), shuffle=bool(train), drop_last=bool(drop_last),
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=False,
        worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(int(seed)),
    )
    if NUM_WORKERS > 0:
        kwargs['prefetch_factor'] = PREFETCH_FACTOR
    return DataLoader(**kwargs)

def build_ram_caches(records, sizes):
    records = list(records)
    sizes = tuple(dict.fromkeys(int(size) for size in sizes))
    image_ids = [record['image_id'] for record in records]
    if len(image_ids) != len(set(image_ids)):
        raise ValueError('Trùng image_id trong RAM cache')
    caches = {
        size: {
            'images': np.empty((len(records), size, size, 3), dtype=np.uint8),
            'masks': np.empty((len(records), size, size), dtype=np.uint8),
            'id_to_index': {image_id: index for index, image_id in enumerate(image_ids)},
        }
        for size in sizes
    }
    for index, record in enumerate(tqdm(records, desc=f'RAM cache {sizes}px')):
        image_np = load_pil_uint8(record['image_path'], 'RGB')
        mask_np = load_pil_uint8(record['mask_path'], 'L')
        if image_np.shape[:2] != mask_np.shape:
            raise ValueError(f"Image/mask lệch kích thước: {record['image_id']}")
        binary_mask = (mask_np > 127).astype(np.uint8)
        for size in sizes:
            resized = cv2.resize(image_np, (size, size), interpolation=cv2.INTER_LINEAR)
            caches[size]['images'][index] = gray_world(resized)
            caches[size]['masks'][index] = cv2.resize(
                binary_mask, (size, size), interpolation=cv2.INTER_NEAREST
            )
    return caches

if CACHE_RESIZED_IN_RAM:
    RAM_CACHE.update(build_ram_caches(train_records, SIZES))
    cache_gib = sum(v['images'].nbytes + v['masks'].nbytes
                    for v in RAM_CACHE.values()) / 2**30
    print(f'Train-only resized uint8 cache: {cache_gib:.2f} GiB | {memory_status()}')

clean_dataset = ISICDataset(train_records[:3], SIZES[0], train=False)
for index in range(len(clean_dataset)):
    x1, y1 = clean_dataset[index]
    x2, y2 = clean_dataset[index]
    assert torch.equal(x1, x2) and torch.equal(y1, y2)
    assert x1.dtype == torch.uint8 and y1.dtype == torch.float32
    assert set(torch.unique(y1).tolist()).issubset({0.0, 1.0})
    assert 0.0 < float(y1.mean()) < 1.0
del clean_dataset, x1, x2, y1, y2
print('Dataset, mask and clean-transform checks: OK')

In [ ]:
class DecoderBlock(nn.Module):
    """Upsample -> ghép skip feature (nếu có) -> Conv -> BatchNorm -> ReLU."""
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:],
                                  mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class MobileNetV2UNet(nn.Module):
    """
    MobileNetV2-UNet:
    Input [B, 3, H, W] -> output raw logits [B, 1, H, W].
    Kiến trúc lấy từ models/mobilenetv2_unet.py của nhánh branch_hoa.
    """
    def __init__(self, pretrained=True, out_channels=1):
        super().__init__()
        weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
        try:
            backbone = models.mobilenet_v2(weights=weights).features
        except Exception as error:
            raise RuntimeError(
                'Không tải được MobileNetV2 ImageNet weights; hãy bật Internet trong Kaggle.'
            ) from error

        # Encoder stages and skip connections from the repository model.
        self.stage0 = backbone[0:2]    # /2, 16 channels
        self.stage1 = backbone[2:4]    # /4, 24 channels
        self.stage2 = backbone[4:7]    # /8, 32 channels
        self.stage3 = backbone[7:14]   # /16, 96 channels
        self.stage4 = backbone[14:18]  # /32, 320 channels

        self.dec4 = DecoderBlock(320, 96, 256)
        self.dec3 = DecoderBlock(256, 32, 128)
        self.dec2 = DecoderBlock(128, 24, 64)
        self.dec1 = DecoderBlock(64, 16, 32)
        self.dec0 = DecoderBlock(32, 0, 16)
        self.final_conv = nn.Conv2d(16, out_channels, kernel_size=1)

    def encoder_modules(self):
        """Cho phép freeze/unfreeze toàn bộ encoder theo hai giai đoạn."""
        return [self.stage0, self.stage1, self.stage2, self.stage3, self.stage4]

    def forward(self, x):
        s0 = self.stage0(x)
        s1 = self.stage1(s0)
        s2 = self.stage2(s1)
        s3 = self.stage3(s2)
        s4 = self.stage4(s3)

        x = self.dec4(s4, s3)
        x = self.dec3(x, s2)
        x = self.dec2(x, s1)
        x = self.dec1(x, s0)
        x = self.dec0(x)
        return self.final_conv(x)  # raw logits, no sigmoid


def unwrap_model(model):
    return model


def to_runtime(model):
    model = model.to(DEVICE)
    if USE_CHANNELS_LAST:
        model = model.to(memory_format=torch.channels_last)
    return model  # intentionally use only cuda:0


def freeze_first_layer_group(model):
    # Với MobileNetV2-UNet, "frozen stage" đóng băng toàn bộ encoder.
    # BatchNorm encoder vẫn được phép cập nhật tham số affine như ở notebook gốc.
    base = unwrap_model(model)
    for parameter in base.parameters():
        parameter.requires_grad = True
    for module in base.encoder_modules():
        for child in module.modules():
            for parameter in child.parameters(recurse=False):
                parameter.requires_grad = isinstance(child, nn.BatchNorm2d)


def unfreeze_with_frozen_bn(model):
    for parameter in model.parameters():
        parameter.requires_grad = True
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            for parameter in module.parameters():
                parameter.requires_grad = False


def set_train_mode(model, freeze_bn):
    model.train()
    if freeze_bn:
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()


def cpu_state_dict(model):
    return {key: value.detach().cpu().clone()
            for key, value in unwrap_model(model).state_dict().items()}


@torch.no_grad()
def refresh_cpu_state_dict(model, destination):
    source = unwrap_model(model).state_dict()
    if tuple(source) != tuple(destination):
        raise RuntimeError('state_dict keys changed during training')
    for key, value in source.items():
        destination[key].copy_(value.detach(), non_blocking=False)


def load_model_state(model, state):
    while state and all(key.startswith('module.') for key in state):
        state = {key.removeprefix('module.'): value for key, value in state.items()}
    unwrap_model(model).load_state_dict(state)


shape_model = MobileNetV2UNet(pretrained=False).eval()
with torch.inference_mode():
    shape_output = shape_model(torch.randn(2, 3, 128, 128))
assert shape_output.shape == (2, 1, 128, 128), shape_output.shape
assert not any(key.startswith('first_layer_group.') for key in shape_model.state_dict())
print(f'Parameters: {sum(p.numel() for p in shape_model.parameters()) / 1e6:.3f}M')
del shape_model, shape_output


In [ ]:
criterion = nn.BCEWithLogitsLoss()
INPUT_DTYPE = torch.float16 if USE_AMP else torch.float32
GPU_MEAN = torch.tensor(IMAGENET_MEAN, device=DEVICE, dtype=INPUT_DTYPE).view(1, 3, 1, 1)
GPU_STD = torch.tensor(IMAGENET_STD, device=DEVICE, dtype=INPUT_DTYPE).view(1, 3, 1, 1)

def prepare_inputs(inputs):
    inputs = inputs.to(DEVICE, non_blocking=True).to(dtype=INPUT_DTYPE)
    inputs.div_(255.0).sub_(GPU_MEAN).div_(GPU_STD)
    if USE_CHANNELS_LAST:
        return inputs.contiguous(memory_format=torch.channels_last)
    return inputs.contiguous()

def per_image_dice(probability, target, threshold=PRED_THRESHOLD):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    denominator = prediction.sum(dims) + target.sum(dims)
    return torch.where(denominator > 0, 2 * intersection / denominator.clamp(min=1),
                       torch.ones_like(denominator))

@torch.inference_mode()
def evaluate_dice(model, loader):
    model.eval()
    dice_sum = torch.zeros((), device=DEVICE)
    sample_count = 0
    for inputs, targets in loader:
        inputs = prepare_inputs(inputs)
        targets = targets.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            probability = torch.sigmoid(model(inputs).float())
        if not bool(torch.isfinite(probability).all()):
            raise FloatingPointError('Model sinh xác suất NaN/Inf')
        dice_sum += per_image_dice(probability, targets).sum()
        sample_count += targets.shape[0]
    if sample_count == 0:
        raise RuntimeError('Holdout loader rỗng')
    return float((dice_sum / sample_count).cpu())

def per_image_scores(probability, target, threshold=PRED_THRESHOLD):
    prediction = (probability > threshold).float()
    dims = (1, 2, 3)
    intersection = (prediction * target).sum(dims)
    prediction_sum = prediction.sum(dims)
    target_sum = target.sum(dims)
    union = prediction_sum + target_sum - intersection
    jaccard = torch.where(union > 0, intersection / union.clamp(min=1),
                           torch.ones_like(union))
    dice = torch.where(prediction_sum + target_sum > 0,
                       2 * intersection / (prediction_sum + target_sum).clamp(min=1),
                       torch.ones_like(union))
    sensitivity = torch.where(target_sum > 0,
                              intersection / target_sum.clamp(min=1),
                              torch.ones_like(target_sum))
    return jaccard, dice, sensitivity

def mask_surface(mask):
    eroded = ndimage.binary_erosion(mask, border_value=0)
    return np.logical_xor(mask, eroded)

def hd95_single(prediction, target):
    prediction, target = prediction.astype(bool), target.astype(bool)
    height, width = target.shape
    if not prediction.any() and not target.any():
        return 0.0
    if not prediction.any() or not target.any():
        return float(math.hypot(height, width))
    prediction_surface = mask_surface(prediction)
    target_surface = mask_surface(target)
    distance_to_target = ndimage.distance_transform_edt(~target_surface)
    distance_to_prediction = ndimage.distance_transform_edt(~prediction_surface)
    distances = np.concatenate([distance_to_target[prediction_surface],
                                distance_to_prediction[target_surface]])
    return float(np.percentile(distances, 95)) if distances.size else 0.0

@torch.inference_mode()
def ensemble_probability(models_list, inputs):
    total = torch.zeros((inputs.shape[0], 1, inputs.shape[2], inputs.shape[3]),
                        device=inputs.device, dtype=torch.float32)
    for model in models_list:
        model.eval()
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
            probability = torch.sigmoid(model(inputs).float())
        if not bool(torch.isfinite(probability).all()):
            raise FloatingPointError('Model sinh xác suất NaN/Inf khi evaluation')
        total += probability
    return total / len(models_list)

@torch.inference_mode()
def evaluate_full(models_list, loader, include_hd95=True):
    sums = {'jaccard': 0.0, 'threshold_jaccard': 0.0,
            'dice': 0.0, 'sensitivity': 0.0}
    hd95_sum, sample_count = 0.0, 0
    for inputs, targets in loader:
        inputs = prepare_inputs(inputs)
        targets_device = targets.to(DEVICE, non_blocking=True)
        probability = ensemble_probability(models_list, inputs)
        jaccard, dice, sensitivity = per_image_scores(probability, targets_device)
        sums['jaccard'] += float(jaccard.sum())
        sums['threshold_jaccard'] += float((jaccard * (jaccard >= JACCARD_CUT)).sum())
        sums['dice'] += float(dice.sum())
        sums['sensitivity'] += float(sensitivity.sum())
        sample_count += targets.shape[0]
        if include_hd95:
            prediction_np = (probability > PRED_THRESHOLD).cpu().numpy()[:, 0]
            target_np = (targets.numpy()[:, 0] > 0.5)
            hd95_sum += sum(hd95_single(prediction, target)
                              for prediction, target in zip(prediction_np, target_np))
    if sample_count == 0:
        raise RuntimeError('Evaluation loader rỗng')
    metrics = {key: value / sample_count for key, value in sums.items()}
    if include_hd95:
        metrics['hd95'] = hd95_sum / sample_count
    return metrics

metric_target = torch.zeros(2, 1, 8, 8, device=DEVICE)
metric_target[0, :, :4] = 1
metric_target[1, :, :, :4] = 1
metric_probability = metric_target.clone()
metric_probability[1] = 0
metric_probability[1, :, :, :2] = 1
assert torch.allclose(per_image_dice(metric_probability, metric_target),
                      torch.tensor([1.0, 2 / 3], device=DEVICE), atol=1e-6)
del metric_target, metric_probability
print('Dice metric check: OK')

In [ ]:
def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def trusted_torch_load(path, map_location='cpu'):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except TypeError:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)

def trainable_parameters(model):
    parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not parameters:
        raise RuntimeError('Không có parameter trainable')
    return parameters

def optimizer_for(model, learning_rate):
    parameters = trainable_parameters(model)
    if USE_FUSED_ADAM:
        try:
            return torch.optim.Adam(parameters, lr=learning_rate,
                                    weight_decay=WEIGHT_DECAY, fused=True)
        except (TypeError, RuntimeError):
            pass
    return torch.optim.Adam(parameters, lr=learning_rate, weight_decay=WEIGHT_DECAY)

def train_batch(model, optimizer, scaler, inputs, targets):
    inputs = prepare_inputs(inputs)
    targets = targets.to(DEVICE, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
        loss = criterion(model(inputs).float(), targets)
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError('Train loss là NaN/Inf')
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(
        (p for group in optimizer.param_groups for p in group['params']),
        GRAD_CLIP_NORM,
    )
    scaler.step(optimizer)
    scaler.update()
    return loss.detach()

def lr_find(model, records, size, stage_seed, freeze_bn):
    set_seed(stage_seed)
    original_state = cpu_state_dict(model)
    optimizer = optimizer_for(model, LR_MIN)
    scaler = make_scaler()
    loader = make_loader(records, size, train=True, seed=stage_seed,
                         batch_size=BATCH_SIZE_BY_SIZE[size], drop_last=False)
    iterator = iter(loader)
    multiplier = (LR_MAX / LR_MIN) ** (1 / max(1, LRF_ITERS - 1))
    learning_rates, losses = [], []
    moving_average, best_loss = 0.0, float('inf')
    for iteration in range(LRF_ITERS):
        try:
            inputs, targets = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            inputs, targets = next(iterator)
        learning_rate = LR_MIN * multiplier ** iteration
        for group in optimizer.param_groups:
            group['lr'] = learning_rate
        set_train_mode(model, freeze_bn)
        try:
            raw_loss = float(train_batch(model, optimizer, scaler, inputs, targets).cpu())
        except FloatingPointError:
            break
        moving_average = LRF_BETA * moving_average + (1 - LRF_BETA) * raw_loss
        smoothed = moving_average / (1 - LRF_BETA ** (iteration + 1))
        if not np.isfinite(smoothed) or (iteration > 5 and smoothed > 4 * best_loss):
            break
        best_loss = min(best_loss, smoothed)
        learning_rates.append(learning_rate)
        losses.append(smoothed)
    del iterator, loader, optimizer, scaler, inputs, targets
    load_model_state(model, original_state)
    del original_state
    gc.collect()
    set_seed(stage_seed)
    learning_rates = np.asarray(learning_rates, dtype=float)
    losses = np.asarray(losses, dtype=float)
    if len(learning_rates) < 3:
        raise RuntimeError('LR Finder không đủ điểm hợp lệ')
    gradient = np.gradient(losses, np.log10(learning_rates))
    steepest_lr = float(learning_rates[int(np.argmin(gradient))])
    min_loss_lr = float(learning_rates[int(np.argmin(losses))])
    cap = MAX_LR_UNFROZEN if freeze_bn else MAX_LR_FROZEN
    selected_lr = max(LR_MIN, min(steepest_lr * LR_SELECTION_FACTOR,
                                  min_loss_lr / 10, cap))
    print(f'LR Finder {size}px: selected={selected_lr:.2e}')
    return selected_lr, learning_rates.tolist(), losses.tolist()

def stlr_lambda(total_steps, cut_frac=CUT_FRAC, ratio=STLR_RATIO):
    cut = max(1, int(total_steps * cut_frac))
    def schedule(step):
        if step < cut:
            progress = step / cut
        else:
            denominator = cut * (1 / cut_frac - 1)
            progress = max(0.0, 1 - (step - cut) / max(1, denominator))
        return (1 + progress * (ratio - 1)) / ratio
    return schedule

def ids_digest(records):
    text = '\n'.join(record['image_id'] for record in records)
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def config_digest(config):
    canonical = json.dumps(config, sort_keys=True, separators=(',', ':'), ensure_ascii=True)
    return hashlib.sha256(canonical.encode('utf-8')).hexdigest()

def save_best_state(best_state, best_path, metadata):
    atomic_torch_save({'model_state': best_state, 'meta': metadata}, best_path)

def run_stage(model, train_subset, holdout_subset, size, epochs, freeze_bn,
              fold_number, stage_name, stage_seed):
    stage_dir = CHECKPOINT_DIR / f'fold_{fold_number}' / f'{size}px_{stage_name}'
    best_path = stage_dir / 'best.pt'
    last_path = stage_dir / 'last.pt'
    contract = {
        'protocol_version': PROTOCOL_VERSION, 'fold': fold_number, 'size': size,
        'stage': stage_name, 'epochs': epochs, 'stage_seed': stage_seed,
        'freeze_bn': freeze_bn, 'batch_size': BATCH_SIZE_BY_SIZE[size],
        'train_ids': ids_digest(train_subset), 'holdout_ids': ids_digest(holdout_subset),
        'selection_metric': 'internal_fold_holdout_dice',
        'dataset_handle': DATASET_HANDLE, 'amp': USE_AMP,
        'channels_last': USE_CHANNELS_LAST,
        'fp32_transposed_conv': FP32_TRANSPOSED_CONV,
        'run_config_sha256': config_digest(RUN_CONFIG),
    }

    saved = None
    if RESUME and last_path.is_file():
        saved = trusted_torch_load(last_path, map_location='cpu')
        if saved.get('contract') != contract:
            raise RuntimeError(f'Checkpoint không khớp config/split: {last_path}')
        if saved.get('complete'):
            best_payload = trusted_torch_load(best_path, map_location='cpu')
            if best_payload.get('meta', {}).get('contract') != contract:
                raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
            load_model_state(model, best_payload['model_state'])
            print(f'[resume] fold={fold_number} {size}px {stage_name}: complete')
            return saved['result']

    holdout_loader = make_loader(holdout_subset, size, train=False, seed=stage_seed)
    if saved is None:
        selected_lr, lrf_lrs, lrf_losses = lr_find(
            model, train_subset, size, stage_seed, freeze_bn
        )
        initial_dice = evaluate_dice(model, holdout_loader)
        best_score, best_epoch = initial_dice, 0
        history = []
        best_state = cpu_state_dict(model)
        best_meta = {
            'epoch': best_epoch, 'dice': best_score, 'selected_lr': selected_lr,
            'contract': contract,
        }
        save_best_state(best_state, best_path, best_meta)
        start_epoch = 0
    else:
        selected_lr = float(saved['selected_lr'])
        lrf_lrs, lrf_losses = saved['lrf_lrs'], saved['lrf_losses']
        best_score = float(saved['best_score'])
        best_epoch = int(saved['best_epoch'])
        history = list(saved['history'])
        best_payload = trusted_torch_load(best_path, map_location='cpu')
        best_meta = best_payload.get('meta', {})
        if best_meta.get('contract') != contract:
            raise RuntimeError(f'Best checkpoint không khớp: {best_path}')
        if int(best_meta['epoch']) >= best_epoch:
            best_epoch = int(best_meta['epoch'])
            best_score = float(best_meta['dice'])
        best_state = best_payload['model_state']
        del best_payload
        start_epoch = int(saved['next_epoch'])
        load_model_state(model, saved['model_state'])
        print(f'[resume] fold={fold_number} {size}px {stage_name}: epoch {start_epoch + 1}')

    steps_per_epoch = math.ceil(len(train_subset) / BATCH_SIZE_BY_SIZE[size])
    optimizer = optimizer_for(model, selected_lr)
    scaler = make_scaler()
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, stlr_lambda(epochs * steps_per_epoch)
    )
    if saved is not None:
        optimizer.load_state_dict(saved['optimizer_state'])
        scheduler.load_state_dict(saved['scheduler_state'])
        scaler.load_state_dict(saved['scaler_state'])
        saved = None
        gc.collect()

    for epoch in range(start_epoch, epochs):
        epoch_started = time.perf_counter()
        train_loader = make_loader(train_subset, size, train=True,
                                   seed=stage_seed + epoch, drop_last=False)
        set_train_mode(model, freeze_bn)
        loss_sum = torch.zeros((), device=DEVICE)
        samples_seen = 0
        for inputs, targets in train_loader:
            loss_sum += train_batch(model, optimizer, scaler, inputs, targets)
            samples_seen += inputs.shape[0]
            scheduler.step()
        del inputs, targets
        train_loss = float((loss_sum / steps_per_epoch).cpu())
        del train_loader
        holdout_dice = evaluate_dice(model, holdout_loader)
        row = {
            'fold': fold_number, 'size': size, 'stage': stage_name,
            'epoch': epoch + 1, 'train_loss': train_loss,
            'holdout_dice': holdout_dice, 'lr': optimizer.param_groups[0]['lr'],
        }
        history.append(row)
        if holdout_dice > best_score:
            best_score, best_epoch = holdout_dice, epoch + 1
            refresh_cpu_state_dict(model, best_state)
        if (epoch + 1) % SAVE_LAST_EVERY == 0 or epoch + 1 == epochs:
            best_meta = {
                'epoch': best_epoch, 'dice': best_score, 'selected_lr': selected_lr,
                'contract': contract,
            }
            save_best_state(best_state, best_path, best_meta)
            progress = {
                'complete': False, 'contract': contract, 'next_epoch': epoch + 1,
                'selected_lr': selected_lr, 'lrf_lrs': lrf_lrs,
                'lrf_losses': lrf_losses, 'best_score': best_score,
                'best_epoch': best_epoch, 'history': history,
                'model_state': unwrap_model(model).state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state': scaler.state_dict(),
            }
            atomic_torch_save(progress, last_path)
            del progress
        seconds = time.perf_counter() - epoch_started
        print(
            f'[fold {fold_number} | {size}px | {stage_name}] '
            f'epoch {epoch + 1:02d}/{epochs} loss={train_loss:.4f} '
            f'holdout_dice={holdout_dice:.4f} best={best_score:.4f} '
            f'time={seconds:.1f}s img/s={samples_seen / seconds:.1f} {memory_status()}'
        )

    load_model_state(model, best_state)
    result = {
        'fold': fold_number, 'size': size, 'stage': stage_name,
        'selected_lr': selected_lr, 'best_epoch': best_epoch,
        'best_holdout_dice': best_score, 'history': history,
        'lrf_lrs': lrf_lrs, 'lrf_losses': lrf_losses,
        'best_path': str(best_path),
    }
    atomic_torch_save({'complete': True, 'contract': contract, 'result': result}, last_path)
    del best_state, holdout_loader, optimizer, scaler, scheduler
    gc.collect()
    return result

In [ ]:
def preflight_batch(size):
    batch_size = BATCH_SIZE_BY_SIZE[size]
    model = to_runtime(MobileNetV2UNet(pretrained=False))
    optimizer = optimizer_for(model, 1e-4)
    scaler = make_scaler()
    inputs = torch.randint(0, 256, (batch_size, 3, size, size), dtype=torch.uint8)
    targets = torch.randint(0, 2, (batch_size, 1, size, size), dtype=torch.float32)
    try:
        loss = float(train_batch(model, optimizer, scaler, inputs, targets).cpu())
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError as error:
        raise RuntimeError(
            f'Batch {batch_size} không vừa GPU ở {size}px; giảm PER_GPU_BATCH_BY_SIZE.'
        ) from error
    finally:
        del model, optimizer, scaler, inputs, targets
        gc.collect()
        torch.cuda.empty_cache()
    print(f'Preflight {size}px batch={batch_size}: loss={loss:.4f}')

for size in SIZES:
    preflight_batch(size)

In [ ]:
RUN_CONFIG = {
    'dataset_handle': DATASET_HANDLE,
    'expected_counts': EXPECTED_COUNTS,
    'official_splits_used': ['Training', 'Validation', 'Test'],
    'official_validation_evaluated': True, 'official_test_evaluated': True,
    'seed': SEED, 'n_folds': N_FOLDS, 'sizes': list(SIZES),
    'frozen_epochs': FROZEN_EPOCHS, 'unfrozen_epochs': UNFROZEN_EPOCHS,
    'batch_size_by_size': BATCH_SIZE_BY_SIZE, 'num_workers': NUM_WORKERS,
    'prefetch_factor': PREFETCH_FACTOR, 'pin_memory': PIN_MEMORY,
    'pretrained': PRETRAINED, 'selection_metric': 'internal_fold_holdout_dice',
    'loss': 'BCEWithLogitsLoss',
    'optimizer': 'Adam(fused)' if USE_FUSED_ADAM else 'Adam',
    'weight_decay': WEIGHT_DECAY, 'grad_clip_norm': GRAD_CLIP_NORM,
    'lr_finder': {'min': LR_MIN, 'max': LR_MAX, 'iterations': LRF_ITERS,
                  'beta': LRF_BETA, 'selection_factor': LR_SELECTION_FACTOR,
                  'max_lr_frozen': MAX_LR_FROZEN,
                  'max_lr_unfrozen': MAX_LR_UNFROZEN},
    'stlr': {'cut_frac': CUT_FRAC, 'ratio': STLR_RATIO},
    'augmentation': 'D4+Affine(rotate44,scale1.0-1.05)+BrightnessContrast0.2',
    'augmentation_only_on_train_subset': True,
    'preprocessing': 'resize+gray_world+imagenet_normalize_on_gpu',
    'amp': USE_AMP, 'channels_last': USE_CHANNELS_LAST,
    'fp32_transposed_conv': FP32_TRANSPOSED_CONV,
    'fused_adam': USE_FUSED_ADAM, 'cache_train_uint8': CACHE_RESIZED_IN_RAM,
    'drop_os_page_cache': DROP_OS_PAGE_CACHE, 'save_last_every': SAVE_LAST_EVERY,
    'prediction_threshold': PRED_THRESHOLD, 'jaccard_cut': JACCARD_CUT,
    'protocol_version': PROTOCOL_VERSION,
}
(RUN_DIR / 'config.json').write_text(
    json.dumps(RUN_CONFIG, indent=2, ensure_ascii=False), encoding='utf-8'
)

def train_fold(fold_number, train_subset, holdout_subset):
    fold_seed = SEED + fold_number - 1
    set_seed(fold_seed)
    first_marker = CHECKPOINT_DIR / f'fold_{fold_number}' / f'{SIZES[0]}px_frozen' / 'last.pt'
    continuing = RESUME and first_marker.is_file()
    model = to_runtime(MobileNetV2UNet(pretrained=PRETRAINED and not continuing))
    stage_results = []
    for size_index, size in enumerate(SIZES):
        freeze_first_layer_group(model)
        stage_results.append(run_stage(
            model, train_subset, holdout_subset, size, FROZEN_EPOCHS, False,
            fold_number, 'frozen', fold_seed + size_index * 1000,
        ))
        unfreeze_with_frozen_bn(model)
        stage_results.append(run_stage(
            model, train_subset, holdout_subset, size, UNFROZEN_EPOCHS, True,
            fold_number, 'unfrozen', fold_seed + size_index * 1000 + 500,
        ))

    holdout_loader = make_loader(holdout_subset, SIZES[-1], train=False, seed=fold_seed)
    final_holdout_dice = evaluate_dice(model, holdout_loader)
    final_path = CHECKPOINT_DIR / f'fold_{fold_number}' / 'mobilenetv2_unet_best.pt'
    metadata = {
        'fold': fold_number, 'fold_seed': fold_seed, 'config': RUN_CONFIG,
        'train_count': len(train_subset), 'holdout_count': len(holdout_subset),
        'train_ids': ids_digest(train_subset), 'holdout_ids': ids_digest(holdout_subset),
        'holdout_dice': final_holdout_dice, 'stage_results': stage_results,
    }
    atomic_torch_save({'model_state': unwrap_model(model).state_dict(),
                       'meta': metadata}, final_path)
    rows = [row for result in stage_results for row in result['history']]
    pd.DataFrame(rows).to_csv(LOG_DIR / f'fold_{fold_number}_history.csv', index=False)
    del model, holdout_loader
    gc.collect()
    torch.cuda.empty_cache()
    return final_path, metadata

fold_checkpoint_paths, fold_metadata = [], []
training_started = time.time()
for fold_number, (train_subset, holdout_subset) in enumerate(folds, start=1):
    final_path = CHECKPOINT_DIR / f'fold_{fold_number}' / 'mobilenetv2_unet_best.pt'
    if RESUME and final_path.is_file():
        payload = trusted_torch_load(final_path, map_location='cpu')
        metadata = payload['meta']
        if (metadata.get('config') != RUN_CONFIG
                or metadata.get('train_ids') != ids_digest(train_subset)
                or metadata.get('holdout_ids') != ids_digest(holdout_subset)):
            raise RuntimeError(f'Fold checkpoint không khớp: {final_path}')
        del payload
        fold_path = final_path
        print(f"[resume] fold {fold_number} complete, Dice={metadata['holdout_dice']:.4f}")
    else:
        print(f'\n===== FOLD {fold_number}/{N_FOLDS}: train={len(train_subset)}, '
              f'holdout={len(holdout_subset)} =====')
        fold_path, metadata = train_fold(fold_number, train_subset, holdout_subset)
    fold_checkpoint_paths.append(str(fold_path))
    fold_metadata.append(metadata)

fold_table = pd.DataFrame([
    {'fold': metadata['fold'], 'train_count': metadata['train_count'],
     'holdout_count': metadata['holdout_count'],
     'internal_holdout_dice': metadata['holdout_dice']}
    for metadata in fold_metadata
])
training_elapsed_hours = (time.time() - training_started) / 3600
fold_table.to_csv(LOG_DIR / 'fold_holdout_dice.csv', index=False)
display(fold_table.style.format({'internal_holdout_dice': '{:.4f}'}))
print(f'Training complete in {training_elapsed_hours:.2f} hours. '
      'Starting official validation/test in the next cell.')
print('Checkpoints:', fold_checkpoint_paths)

## Đánh giá official validation và test

Single model được khóa bằng internal fold-holdout Dice trước khi mở official validation/test. Validation chỉ dùng để báo cáo từng fold và equal-weight probability ensemble; test báo cáo single đã khóa và ensemble. HD95 có đơn vị pixel trên mask đã resize `256×256`.

In [ ]:
selected_single_index = int(np.argmax([meta['holdout_dice'] for meta in fold_metadata]))
selected_single_fold = selected_single_index + 1
selected_checkpoint_path = CHECKPOINT_DIR / 'mobilenetv2_unet_selected_single_best.pt'
runtime_models = []
for index, checkpoint_path in enumerate(fold_checkpoint_paths):
    payload = trusted_torch_load(checkpoint_path, map_location='cpu')
    if index == selected_single_index:
        selected_meta = dict(payload['meta'])
        selected_meta['selection'] = 'highest internal fold-holdout Dice'
        selected_meta['selected_fold'] = selected_single_fold
        atomic_torch_save({'model_state': payload['model_state'], 'meta': selected_meta},
                          selected_checkpoint_path)
    model = to_runtime(MobileNetV2UNet(pretrained=False))
    load_model_state(model, payload['model_state'])
    runtime_models.append(model)
    del payload
gc.collect()
torch.cuda.empty_cache()

validation_loader = make_loader(official_val_records, SIZES[-1], train=False, seed=SEED)
validation_rows = []
for index, model in enumerate(runtime_models, start=1):
    validation_rows.append({
        'model': f'BestDice fold {index}',
        **evaluate_full([model], validation_loader, include_hd95=True),
    })
validation_rows.append({
    'model': 'Equal-weight ensemble',
    **evaluate_full(runtime_models, validation_loader, include_hd95=True),
})
validation_table = pd.DataFrame(validation_rows)
validation_table.to_csv(LOG_DIR / 'official_validation_metrics.csv', index=False)
del validation_loader

test_loader = make_loader(test_records, SIZES[-1], train=False, seed=SEED)
test_rows = [
    {'model': f'Single BestDice fold {selected_single_fold}',
     **evaluate_full([runtime_models[selected_single_index]], test_loader, include_hd95=True)},
    {'model': 'Equal-weight ensemble',
     **evaluate_full(runtime_models, test_loader, include_hd95=True)},
]
test_table = pd.DataFrame(test_rows)
test_table.to_csv(LOG_DIR / 'official_test_metrics.csv', index=False)
del test_loader

summary = {
    'config': RUN_CONFIG,
    'sample_counts': {'train': len(train_records),
                      'validation': len(official_val_records),
                      'test': len(test_records)},
    'elapsed_training_hours': training_elapsed_hours,
    'fold_checkpoints': fold_checkpoint_paths,
    'fold_internal_holdout': fold_table.to_dict(orient='records'),
    'selected_single_fold_by_internal_cv_dice': selected_single_fold,
    'selected_single_checkpoint': str(selected_checkpoint_path),
    'official_validation': validation_table.to_dict(orient='records'),
    'official_test': test_table.to_dict(orient='records'),
    'hd95_unit': f'pixel at {SIZES[-1]}x{SIZES[-1]} resolution',
}
summary_path = RUN_DIR / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
metric_format = {column: '{:.4f}' for column in validation_table.columns if column != 'model'}
display(validation_table.style.format(metric_format))
display(test_table.style.format({column: '{:.4f}' for column in test_table.columns
                                 if column != 'model'}))
print('Selected single fold:', selected_single_fold)
print('Selected checkpoint:', selected_checkpoint_path)
print('Summary:', summary_path)


## Kết quả được tạo

- Ba checkpoint fold-final để resume/tái tạo ensemble và một checkpoint single được chọn hoàn toàn bằng internal CV Dice.
- Lịch sử train từng fold, bảng internal holdout, official validation và official test trong `logs`.
- `summary.json` chứa cấu hình, split counts, checkpoint paths và toàn bộ metrics.
- Official validation/test không tham gia LR Finder, augmentation, early stopping, chọn fold, threshold hoặc ensemble weight.